In [1]:
import pandas as pd
import xgboost as xgb
import json
import os
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

DATA_PATH = "data/bus_delay_dataset_final.parquet"
OUTPUT_DIR = "output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f"Total rows loaded: {len(df):,}")
df.head()

Total rows loaded: 5,317,023


,route_id,stop_id,stop_lat,stop_lon,stop_sequence,day_of_week,hour_of_day,holiday_flag,delay_minutes,arrival_seconds,departure_seconds,route_id_freq,stop_id_freq,trip_id_freq,normalized_stop_position,prev_delay,trip_id
0,0,3127,28.553810,77.058640,0,6,5,0,-9.82,19380,19380,910,613,26,0.000000,0.00,0_05_23
1,0,2620,28.551636,77.056856,1,1,5,0,13.94,19451,19451,910,685,26,0.038462,-9.82,0_05_23
2,0,2020,28.558792,77.042804,2,4,5,0,-6.37,19832,19832,910,211,26,0.076923,13.94,0_05_23
3,0,2021,28.552933,77.035183,3,1,5,0,-3.64,20069,20069,910,498,26,0.115385,-6.37,0_05_23
4,0,4120,28.542403,77.029792,4,4,5,0,1.90,20377,20377,910,253,26,0.153846,-3.64,0_05_23


In [3]:
TARGET = 'delay_minutes'
FEATURES = [col for col in df.columns if col not in [TARGET, 'trip_id']]

print(f"Selected {len(FEATURES)} features: {FEATURES}")

missing_cols = [col for col in FEATURES + [TARGET] if col not in df.columns]
if missing_cols:
    print(f"WARNING: Missing columns in dataset: {missing_cols}")

X = df[FEATURES]
y = df[TARGET]

Selected 15 features: ['route_id', 'stop_id', 'stop_lat', 'stop_lon', 'stop_sequence', 'day_of_week', 'hour_of_day', 'holiday_flag', 'arrival_seconds', 'departure_seconds', 'route_id_freq', 'stop_id_freq', 'trip_id_freq', 'normalized_stop_position', 'prev_delay']


In [4]:
import numpy as np

# Splitting based on trip_id to prevent data leakage of sequential features like prev_delay
np.random.seed(42)
unique_trips = df['trip_id'].unique()
np.random.shuffle(unique_trips)

train_trips_bound = int(len(unique_trips) * 0.70)
val_trips_bound = int(len(unique_trips) * 0.85)

train_trips = set(unique_trips[:train_trips_bound])
val_trips = set(unique_trips[train_trips_bound:val_trips_bound])
test_trips = set(unique_trips[val_trips_bound:])

# Filter data securely without 'trip_id' making it into the features
train_mask = df['trip_id'].isin(train_trips)
val_mask = df['trip_id'].isin(val_trips)
test_mask = df['trip_id'].isin(test_trips)

X_train, y_train = df.loc[train_mask, FEATURES], df.loc[train_mask, TARGET]
X_val, y_val = df.loc[val_mask, FEATURES], df.loc[val_mask, TARGET]
X_test, y_test = df.loc[test_mask, FEATURES], df.loc[test_mask, TARGET]

print(f"Train set: {len(X_train):,} rows ({len(X_train)/len(df):.1%})")
print(f"Validation set: {len(X_val):,} rows ({len(X_val)/len(df):.1%})")
print(f"Test set: {len(X_test):,} rows ({len(X_test)/len(df):.1%})")


Train set: 3,718,489 rows (69.9%)
Validation set: 801,447 rows (15.1%)
Test set: 797,087 rows (15.0%)


In [5]:
#debug for leakage
print(len(train_trips & val_trips))
print(len(train_trips & test_trips))
print(len(val_trips & test_trips))

0
0
0


In [6]:
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='mae',
    learning_rate=0.05,
    n_estimators=600,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    n_jobs=-1,
    gamma=1,
    reg_lambda=1,
    reg_alpha=0.1,
    early_stopping_rounds=50
)

In [7]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print(f"\nTraining complete, Best iteration: {model.best_iteration}")

[0]	validation_0-mae:7.49845
[50]	validation_0-mae:7.49863
[54]	validation_0-mae:7.49865

Training complete, Best iteration: 4


In [8]:
model.get_booster().get_score()

{'route_id': 755.0,
 'stop_id': 840.0,
 'stop_lat': 928.0,
 'stop_lon': 907.0,
 'stop_sequence': 756.0,
 'day_of_week': 853.0,
 'hour_of_day': 535.0,
 'holiday_flag': 140.0,
 'arrival_seconds': 1618.0,
 'departure_seconds': 425.0,
 'route_id_freq': 757.0,
 'stop_id_freq': 798.0,
 'trip_id_freq': 546.0,
 'normalized_stop_position': 957.0,
 'prev_delay': 1480.0}

In [9]:
import numpy as np

# Check correlation between prev_delay and delay_minutes in the SAME trip
corr = df[['prev_delay', 'delay_minutes']].corr()
print(corr)

# Also check: what fraction of trips have prev_delay = 0
zero_prev = (df['prev_delay'] == 0).mean()
print(f"\nFraction of rows with prev_delay == 0.0: {zero_prev:.2%}")

# And check: how spread is delay_minutes?
print(f"\ndelay_minutes stats:\n{df['delay_minutes'].describe()}")


               prev_delay  delay_minutes
prev_delay       1.000000       0.000058
delay_minutes    0.000058       1.000000

Fraction of rows with prev_delay == 0.0: 2.51%

delay_minutes stats:
count    5.317023e+06
mean    -7.772789e-04
std      8.659070e+00
min     -1.500000e+01
25%     -7.500000e+00
50%      0.000000e+00
75%      7.500000e+00
max      1.500000e+01
Name: delay_minutes, dtype: float64
